# Reddit Sentiment Analysis — v2: Context-Aware Model (LSTM)

**Why this notebook exists:** the previous model (TF-IDF + Dense Neural Network) is a "bag of words" —
it looks at *which* words appear, but not the *order* they appear in. That caused real, confirmed errors:

- `"It was a good day but I am tired"` → predicted Negative (should lean Neutral/mixed) because
  `"but"` was deleted as a stopword before the model ever saw it, so the contrast was lost.
- `"Happy Singh is sad"` → predicted Neutral, because `Happy` and `sad` are just averaged as two
  opposite emotion words, with no idea that `Happy Singh` is a person's *name*.

**What this notebook fixes:**
1. Preprocessing no longer deletes negation/contrast words (`not`, `no`, `never`, `but`, `however`, etc.)
2. The model is now an **Embedding + Bidirectional LSTM**, which reads the sentence *in order* —
   left to right — instead of as an unordered bag of words. This lets it learn patterns like
   "when 'but' shows up, the words after it usually matter more."

**Honest limitation that remains:** the name-collision case (`"Happy Singh is sad"`) is a genuinely
harder problem — recognizing that "Happy" is a proper noun, not an emotion word, needs Named Entity
Recognition (NER), which is a separate NLP task. This notebook does **not** claim to solve that; it
solves negation/contrast handling, which covers the large majority of real sentences.

Run every cell top to bottom. This will take a few minutes on the LSTM training step even on CPU.


## Step 1 — Imports

In [ ]:
import re
import json
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import nltk
from nltk.corpus import stopwords

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from sklearn.preprocessing import label_binarize

nltk.download('stopwords')

print("All libraries loaded.")


## Step 2 — Load the dataset

Same dataset as before (`reddit_sentiment_dataset_v9.csv`) — we are NOT throwing away your data,
just fixing how it's cleaned and what kind of model learns from it.

In [ ]:
df = pd.read_csv("dataset/reddit_sentiment_dataset_v9.csv")
df = df.dropna(subset=["text", "sentiment"])
print("Total rows:", len(df))
print(df["sentiment"].value_counts())
df.head()


## Step 3 — Exploratory Data Analysis (EDA)

Quick visual check of the class balance before we train, so we know what we're working with.

In [ ]:
counts = df["sentiment"].value_counts()
plt.figure(figsize=(5,4))
plt.bar(counts.index, counts.values, color=["#7193FF", "#C9A227", "#2FB380"])
plt.title("Class Distribution")
plt.ylabel("Number of rows")
plt.show()

print((counts / counts.sum() * 100).round(1))


## Step 4 — Fixed Text Cleaning (the key fix)

**What changed vs. the old `preprocessing.py`:** the old version used NLTK's full English stopword
list, which includes negation and contrast words. We remove those specific words FROM the stopword
list, so they survive cleaning and stay visible to the model.

Everything else (lowercase, strip URLs, strip punctuation, tokenize) stays the same.

In [ ]:
# Words that carry negation or contrast meaning -- we deliberately KEEP these,
# even though NLTK's default stopword list would normally remove them.
NEGATION_AND_CONTRAST_WORDS = {
    "not", "no", "nor", "never",
    "but", "however", "although", "though", "yet", "still",
    "don't", "doesn't", "didn't", "isn't", "wasn't", "aren't", "weren't",
    "won't", "wouldn't", "can't", "cannot", "couldn't", "shouldn't",
    "hasn't", "haven't", "hadn't",
    "dont", "doesnt", "didnt", "isnt", "wasnt", "arent", "werent",
    "wont", "wouldnt", "cant", "couldnt", "shouldnt", "hasnt", "havent", "hadnt",
}

_ALL_STOPWORDS = set(stopwords.words("english"))
KEPT_STOPWORDS = _ALL_STOPWORDS - NEGATION_AND_CONTRAST_WORDS  # words we still remove


def clean_text(text: str) -> str:
    """
    Cleans raw text but PRESERVES negation/contrast words, unlike the old version.
    Order of words is preserved (we don't sort or bag them) because this feeds
    a sequence model that cares about word order.
    """
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)
    text = re.sub(r"[^a-zA-Z'\s]", "", text)   # keep apostrophes so "don't" survives as one token
    words = text.split()
    words = [w for w in words if w not in KEPT_STOPWORDS]
    return " ".join(words)


# quick sanity check against the exact sentences that were failing before
test_sentences = [
    "It was a good day but I am tired",
    "Happy Singh is sad",
    "This is not good at all",
]
for s in test_sentences:
    print(f"{s!r}  -->  {clean_text(s)!r}")


In [ ]:
df["clean_text"] = df["text"].apply(clean_text)
df = df[df["clean_text"].str.strip() != ""]
print("Rows after cleaning (empty rows dropped):", len(df))
df[["text", "clean_text", "sentiment"]].head()


## Step 5 — Data Augmentation, part 1: negation

The dataset has very few explicit negation examples ("not good", "don't hate"), so the model
never learned that "not" flips the meaning of the word after it. We fix this by generating
synthetic negated examples with the correct (flipped) label.

In [ ]:
import random

POSITIVE_WORDS = ["good", "great", "happy", "love", "amazing", "awesome", "nice",
                   "fantastic", "well", "fine", "okay", "healthy", "right"]
NEGATIVE_WORDS = ["terrible", "awful", "horrible", "worst",
                   "poor", "sick", "unwell", "ill"]

TEMPLATES = [
    "this is not {word} at all",
    "i am not {word}",
    "i am not feeling {word}",
    "i don't feel {word}",
]

def make_negated_examples(n=1500):
    rows = []
    for _ in range(n):
        template = random.choice(TEMPLATES)
        if random.random() < 0.5:
            word = random.choice(POSITIVE_WORDS)
            text = template.format(word=word)
            label = "negative"
        else:
            word = random.choice(NEGATIVE_WORDS)
            text = template.format(word=word)
            label = "positive"
        rows.append({"text": text, "sentiment": label})
    return pd.DataFrame(rows)

negation_df = make_negated_examples(1500)
negation_df["clean_text"] = negation_df["text"].apply(clean_text)

df = pd.concat([df, negation_df], ignore_index=True)
print("New total rows after negation augmentation:", len(df))
print(df["sentiment"].value_counts())


### Step 5b — Negation for verb-style emotion words ("don't hate", "don't love")

"hate" and "love" are VERBS, not adjectives -- so they don't fit the "this is not {word} at all"
template grammatically ("this is not hate at all" reads oddly). A previous round strongly
reinforced "hate" as standalone-negative (Step 6), which ended up overpowering negation for this
specific word -- "I don't hate this, actually" started being misclassified as negative. This adds
grammatically correct negation examples specifically for these two verb-style words.

In [ ]:
VERB_EMOTION_TEMPLATES_NEG = [
    "i don't {word} this",
    "i don't {word} this at all",
    "honestly i don't {word} this",
    "i don't {word} this, actually",
    "i actually don't {word} this",
]

def make_verb_negation_examples(n=400):
    rows = []
    for _ in range(n):
        template = random.choice(VERB_EMOTION_TEMPLATES_NEG)
        if random.random() < 0.5:
            text = template.format(word="hate")
            label = "positive"   # "don't hate" = not hating = mildly positive/approving
        else:
            text = template.format(word="love")
            label = "negative"   # "don't love" = not loving = mildly negative/lukewarm
        rows.append({"text": text, "sentiment": label})
    return pd.DataFrame(rows)

verb_negation_df = make_verb_negation_examples(400)
verb_negation_df["clean_text"] = verb_negation_df["text"].apply(clean_text)

df = pd.concat([df, verb_negation_df], ignore_index=True)
print("New total rows after verb-negation augmentation:", len(df))
print(df["sentiment"].value_counts())


## Step 6 — Data Augmentation, part 2: reinforcing core emotion vocabulary

Even beyond negation, many individual emotion words are inconsistently labeled in the source
dataset -- e.g. 61% of rows containing the word "sad" are labeled neutral, not negative (same
pattern holds for "angry", "happy", "excited", "miserable", and others). This is a data quality
issue (noisy/inconsistent labeling, common in social-media sentiment datasets), not a model or
architecture problem. We add clear, unambiguous synthetic examples for these words so the model
has enough clean signal to learn their true polarity, rather than being swayed by inconsistently
labeled real-world examples.

In [ ]:
NEG_EMOTION_WORDS = ["sad", "bad", "angry", "upset", "depressed", "miserable",
                      "heartbroken", "hurt", "annoyed", "furious", "hate"]
POS_EMOTION_WORDS = ["happy", "excited", "joyful", "delighted", "thrilled",
                      "grateful", "content", "cheerful", "proud", "love"]

EMOTION_TEMPLATES = [
    "i feel {word}",
    "i am {word}",
    "this makes me {word}",
    "she seems {word}",
    "he was {word} today",
    "everyone is {word}",
    "i'm so {word} right now",
    "feeling {word}",
    # Greeting + filler patterns -- added because casual "hi ... today"
    # style sentences were found to override the emotion word's own
    # signal (e.g. "hi angry today" was predicted neutral even though
    # "angry" alone correctly predicts negative). These templates give
    # the model direct examples of emotion words appearing inside this
    # exact casual sentence shape.
    "hi i am {word} today",
    "hi i am feeling {word} today",
    "hello i am {word} today",
    "hey i feel {word} today",
    "hi everyone i am {word} today",
    "hi i am {word} right now",
]

def make_emotion_examples(n_per_class=1000):
    rows = []
    for _ in range(n_per_class):
        word = random.choice(NEG_EMOTION_WORDS)
        template = random.choice(EMOTION_TEMPLATES)
        rows.append({"text": template.format(word=word), "sentiment": "negative"})
    for _ in range(n_per_class):
        word = random.choice(POS_EMOTION_WORDS)
        template = random.choice(EMOTION_TEMPLATES)
        rows.append({"text": template.format(word=word), "sentiment": "positive"})
    return pd.DataFrame(rows)

emotion_df = make_emotion_examples(1000)
emotion_df["clean_text"] = emotion_df["text"].apply(clean_text)

df = pd.concat([df, emotion_df], ignore_index=True)
print("New total rows after emotion-word augmentation:", len(df))
print(df["sentiment"].value_counts())


## Step 7 — Data Augmentation, part 3: plain self-introductions are neutral

Investigation found that after NER removes a person's name, a plain introduction like
"my name is X" reduces to just the single word "name" (once stopwords are removed) -- and
"name" appeared in only 6 rows of the original dataset, an unreliable sample the model latched
onto. A plain self-introduction genuinely has no sentiment, so we reinforce that directly.

In [ ]:
NEUTRAL_INTRO_TEMPLATES = [
    "my name is",
    "hi my name is",
    "hello my name is",
    "this is",
    "call me",
    "i am known as",
    "you can call me",
    "hi i am",
]

def make_neutral_intro_examples(n=400):
    rows = []
    for _ in range(n):
        text = random.choice(NEUTRAL_INTRO_TEMPLATES)
        rows.append({"text": text, "sentiment": "neutral"})
    return pd.DataFrame(rows)

neutral_intro_df = make_neutral_intro_examples(400)
neutral_intro_df["clean_text"] = neutral_intro_df["text"].apply(clean_text)

df = pd.concat([df, neutral_intro_df], ignore_index=True)
print("New total rows after neutral-introduction augmentation:", len(df))
print(df["sentiment"].value_counts())


## Step 8 — Tokenization + Padding (sequence-based, not TF-IDF)

Instead of TF-IDF (which produces one big "word count" vector with no order), we now:
1. Build a vocabulary (`Tokenizer`) mapping each word to an integer ID.
2. Convert every sentence into a **sequence** of those integer IDs, in the original word order.
3. Pad every sequence to the same length (`MAX_LEN`) so they can be batched together —
   short sentences get zeros added at the end, long ones get cut off.

This is what lets the LSTM "read" the sentence in order, instead of just seeing a word count.

In [ ]:
VOCAB_SIZE = 10000     # keep the top 10,000 most frequent words
MAX_LEN = 40           # max words per sentence (pad/truncate to this length)

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(df["clean_text"])

sequences = tokenizer.texts_to_sequences(df["clean_text"])
X = pad_sequences(sequences, maxlen=MAX_LEN, padding="post", truncating="post")

print("Example sentence:", df["clean_text"].iloc[0])
print("As integer sequence:", sequences[0])
print("Padded shape:", X.shape)


## Step 9 — Encode labels and split into train/test

In [ ]:
encoder = LabelEncoder()
y = encoder.fit_transform(df["sentiment"])   # negative=0, neutral=1, positive=2 (alphabetical)
print("Classes:", list(encoder.classes_))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Train size:", X_train.shape[0], " Test size:", X_test.shape[0])


## Step 10 — Class weights

Your dataset is imbalanced (50% negative, 33% positive, 17% neutral). Class weighting tells the
model "mistakes on the neutral class cost more" during training, so it doesn't just lazily predict
the majority class all the time.

In [ ]:
class_weights_arr = compute_class_weight(
    class_weight="balanced", classes=np.unique(y_train), y=y_train
)
class_weights = dict(enumerate(class_weights_arr))
print("Class weights:", class_weights)


## Step 11 — Build the LSTM model

Architecture:
- `Embedding` — turns each word ID into a learned dense vector (words with similar meaning end up
  with similar vectors, learned automatically during training).
- `Bidirectional(LSTM)` — reads the sentence forward AND backward, so it has context from both
  directions when deciding what a word means in this particular sentence.
- `Dropout` — randomly ignores some neurons during training, to prevent overfitting.
- `Dense(3, softmax)` — final layer, outputs 3 probabilities (negative/neutral/positive) summing to 100%.

In [ ]:
EMBEDDING_DIM = 64

model = Sequential([
    Embedding(input_dim=VOCAB_SIZE, output_dim=EMBEDDING_DIM, input_length=MAX_LEN),
    Bidirectional(LSTM(64, return_sequences=False)),
    Dropout(0.4),
    Dense(32, activation="relu"),
    Dropout(0.3),
    Dense(3, activation="softmax"),
])

model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()


## Step 12 — Train

In [ ]:
early_stop = EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    validation_split=0.15,
    epochs=25,
    batch_size=64,
    class_weight=class_weights,
    callbacks=[early_stop],
    verbose=1,
)


## Step 13 — Evaluate on the held-out test set

In [ ]:
probs = model.predict(X_test)
y_pred = np.argmax(probs, axis=1)

test_accuracy = float((y_pred == y_test).mean())
print("Test accuracy:", round(test_accuracy, 4))

report = classification_report(y_test, y_pred, target_names=encoder.classes_, output_dict=True)
print(classification_report(y_test, y_pred, target_names=encoder.classes_))

cm = confusion_matrix(y_test, y_pred)
print("Confusion matrix:\n", cm)


In [ ]:
# Confusion matrix heatmap
plt.figure(figsize=(5,4))
plt.imshow(cm, cmap="Oranges")
plt.xticks(range(len(encoder.classes_)), encoder.classes_)
plt.yticks(range(len(encoder.classes_)), encoder.classes_)
plt.xlabel("Predicted")
plt.ylabel("Actual")
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j], ha="center", va="center")
plt.title("Confusion Matrix (v2 LSTM model)")
plt.colorbar()
plt.show()


In [ ]:
# ROC curve (one-vs-rest), needed for the Performance page
y_test_bin = label_binarize(y_test, classes=[0, 1, 2])
roc_data = {}
for i, label in enumerate(encoder.classes_):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], probs[:, i])
    roc_data[label] = {"fpr": fpr.tolist(), "tpr": tpr.tolist(), "auc": float(auc(fpr, tpr))}
    print(f"{label}: AUC = {roc_data[label]['auc']:.3f}")


## Step 14 — Sanity check against the sentences that were failing

This is the real test: did the fix actually work on the exact examples that were wrong before?

In [ ]:
def predict_one(text):
    cleaned = clean_text(text)
    seq = tokenizer.texts_to_sequences([cleaned])
    padded = pad_sequences(seq, maxlen=MAX_LEN, padding="post", truncating="post")
    p = model.predict(padded, verbose=0)[0]
    pred_label = encoder.inverse_transform([np.argmax(p)])[0]
    return pred_label, {label: round(float(prob)*100, 1) for label, prob in zip(encoder.classes_, p)}

for s in [
    "It was a good day but I am tired",
    "This is not good at all",
    "I don't hate this, actually",
    "Bro really said trust me and then did the single most untrustworthy thing possible",
    "Maruti Suzuki is a bad company",
    "my name is",
    "sad",
    "bad",
]:
    label, probs_dict = predict_one(s)
    print(f"{s!r}\n  --> {label.upper()}  {probs_dict}\n")


## Step 15 — Save all artifacts

These are the new files your Streamlit app will load. Note the names are different (`_v2`) from
the old TF-IDF artifacts, so you don't overwrite the old ones by accident until you're sure the
new model works well.

In [ ]:
model.save("sentiment_model_v2.keras")

with open("tokenizer_v2.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

with open("label_encoder_v2.pkl", "wb") as f:
    pickle.dump(encoder, f)

# dataset stats (used by the Dashboard page)
char_lengths = df["text"].str.len()
word_counts_len = df["clean_text"].str.split().apply(len)

def top_words_for_class(label, n=15):
    text_blob = " ".join(df[df["sentiment"] == label]["clean_text"])
    from collections import Counter
    counts = Counter(text_blob.split())
    return [[w, c] for w, c in counts.most_common(n)]

dataset_stats = {
    "total_rows": int(len(df)),
    "class_counts": df["sentiment"].value_counts().to_dict(),
    "top_words_by_class": {
        label: top_words_for_class(label) for label in encoder.classes_
    },
    "char_length_mean": float(char_lengths.mean()),
    "char_length_median": float(char_lengths.median()),
    "char_length_bins": list(np.histogram(char_lengths, bins=20)[1]),
    "char_length_hist": list(map(int, np.histogram(char_lengths, bins=20)[0])),
    "word_count_mean": float(word_counts_len.mean()),
    "word_count_median": float(word_counts_len.median()),
    "word_count_bins": list(np.histogram(word_counts_len, bins=20)[1]),
    "word_count_hist": list(map(int, np.histogram(word_counts_len, bins=20)[0])),
}

metrics_json = {
    "test_accuracy": test_accuracy,
    "classification_report": report,
    "confusion_matrix": cm.tolist(),
    "labels": list(encoder.classes_),
    "roc_curve": roc_data,
    "training_history": {
        "accuracy": [float(x) for x in history.history["accuracy"]],
        "val_accuracy": [float(x) for x in history.history["val_accuracy"]],
        "loss": [float(x) for x in history.history["loss"]],
        "val_loss": [float(x) for x in history.history["val_loss"]],
    },
    "dataset_stats": dataset_stats,
    "vocab_size": VOCAB_SIZE,
    "total_training_rows": int(len(df)),
    "max_len": MAX_LEN,
}

with open("precomputed_metrics_v2.json", "w") as f:
    json.dump(metrics_json, f)

print("Saved: sentiment_model_v2.keras, tokenizer_v2.pkl, label_encoder_v2.pkl, precomputed_metrics_v2.json")
print("Test accuracy:", round(test_accuracy, 4))


## Next step — after running this notebook

Once this finishes and you have the 4 new files (`sentiment_model_v2.keras`, `tokenizer_v2.pkl`,
`label_encoder_v2.pkl`, `precomputed_metrics_v2.json`) sitting next to this notebook:

1. Copy all 4 files into your project's `models/` folder.
2. Ask me for the updated `config.py`, `utils/preprocessing.py`, `utils/loader.py`, and
   `utils/predictor.py` — these need small changes to load a tokenizer + padded sequences
   instead of a TF-IDF vectorizer. I'll give you the exact drop-in replacements.
3. Restart the Streamlit app and re-test the same sentences that were failing before.
